# The split that explains the most variance

A support line logs how long each call waits before an agent answers, in whole minutes.
Most calls are account questions that an agent picks up within a few minutes. The rest need
a specialist and wait about twenty minutes. The team wants one cutoff that separates short
waits from long ones as cleanly as possible, and the log keeps only a histogram: the number
of calls that waited 0 minutes, 1 minute, and so on. You will compute the mean and variance
from the histogram, the between-group variance at every cutoff, and the best cutoff.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Waits of 5,000 calls in whole minutes, from 0 to 39: account questions wait a few
# minutes, calls that need a specialist wait about twenty.
rng = np.random.default_rng(7)
account = rng.gamma(2.0, 1.5, 3_500)
specialist = rng.normal(22, 5, 1_500)
waits = np.clip(np.round(np.r_[account, specialist]), 0, 39).astype(int)
counts = np.bincount(waits, minlength=40)       # counts[k] = calls that waited k minutes
print(f"{counts.sum()} calls, waits from 0 to {len(counts) - 1} minutes")
print(counts)

## Part 1 (coding task): Mean and variance from a histogram

Write `pmf_mean_var(counts)`. Entry `counts[k]` is the number of calls that waited exactly
`k` minutes, for `k` from 0 to `len(counts) - 1`. Return three things: the probability mass
function of the wait (an array of the same length that adds up to 1), the mean wait and the
variance of the wait. Work from the counts alone, since the checks pass histograms with no
list of waits behind them.

In [ ]:
def pmf_mean_var(counts):
    """PMF, mean and variance of a value k that occurs counts[k] times."""
    counts = np.asarray(counts, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
pmf, mean, var = pmf_mean_var([0, 2, 0, 2])
assert np.allclose(pmf, [0, 0.5, 0, 0.5]), f"pmf {pmf}: divide each count by the total number of calls"
assert np.isclose(mean, 2.0), f"mean {mean}: the waits are 1, 1, 3 and 3, so the mean is 2"
assert np.isclose(var, 1.0), f"variance {var}: every wait is 1 minute from the mean 2, so the variance is 1"
pmf, mean, var = pmf_mean_var([3, 1])
assert np.isclose(mean, 0.25), f"mean {mean} for the waits 0, 0, 0 and 1, expected 0.25"
assert np.isclose(var, 0.1875), f"variance {var} for the waits 0, 0, 0 and 1, expected 0.1875"
pmf, mean, var = pmf_mean_var(counts)
assert np.isclose(pmf.sum(), 1.0), "the PMF should add up to 1"
assert abs(mean - 8.653) < 1e-6, f"mean wait of the calls {mean:.4f}, expected 8.653"
assert abs(var - 86.125791) < 1e-6, f"variance of the waits {var:.4f}, expected 86.1258"
print("Part 1 passed")

## Part 2 (coding task): Between-group variance at every cutoff

A cutoff $t$ splits the calls into a short group, the waits below $t$, and a long group,
the waits of $t$ or more. Write `between_variance(counts)`. It returns an array `b` of length
`len(counts) + 1`, where `b[t]` is the between-group variance at the cutoff $t$:

$$
w_0(\mu_0 - \mu)^2 + w_1(\mu_1 - \mu)^2 ,
$$

with $w_0$ and $w_1$ the shares of the calls in the short and long groups, $\mu_0$ and
$\mu_1$ their mean waits, and $\mu$ the mean of all the calls. A cutoff that leaves a group
empty splits nothing, and `b[t]` is 0 there.

A loop over the cutoffs is fine here. Running totals (`np.cumsum`) give every cutoff at once.

In [ ]:
def between_variance(counts):
    """b[t] = between-group variance when waits below t form one group, for t = 0..len(counts)."""
    counts = np.asarray(counts, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
b = between_variance([1, 1])
assert len(b) == 3, f"expected one entry for each cutoff t = 0, 1, 2, got {len(b)}"
assert np.allclose(b, [0, 0.25, 0]), f"{b}: waits 0 and 1 split at t = 1 into two groups of one call"
b = between_variance([2, 0, 1, 1])
assert np.allclose(b, [0, 1.5625, 1.5625, 49 / 48, 0]), f"{b}: waits 0, 0, 2, 3, with the wait t itself in the long group"
b = between_variance([2, 1, 0, 0])
assert np.all(np.isfinite(b)), "an empty group should give 0, not nan"
assert np.allclose(b, [0, 2 / 9, 0, 0, 0]), f"{b}: waits 0, 0, 1, and cutoffs 2 and 3 leave the long group empty"
b = between_variance(counts)
assert np.isclose(b.max(), 76.678264125), f"largest between-group variance on the calls {b.max():.4f}, expected 76.6783"
print("Part 2 passed")

## Part 3 (coding task): The best cutoff, and the variance left inside the groups

Write `best_split(counts)`. It returns `(t, var_short, var_long, between)`: the cutoff with
the largest between-group variance (the smallest such $t$ if several tie), the variance of
the waits inside the short group and inside the long group at that cutoff, and the
between-group variance there. The check then combines the two group variances into the
within-group variance $w_0\sigma_0^2 + w_1\sigma_1^2$.

In [ ]:
def best_split(counts):
    """(t, var_short, var_long, between) at the cutoff with the largest between-group variance."""
    counts = np.asarray(counts, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
t, v0, v1, between = best_split([2, 0, 1, 1])
assert t == 1, f"cutoff {t}: t = 1 and t = 2 tie on the waits 0, 0, 2, 3, so return the smaller"
assert np.isclose(v0, 0.0) and np.isclose(v1, 0.25), f"group variances {v0}, {v1}: the groups are 0, 0 and 2, 3"
assert np.isclose(between, 1.5625), f"between-group variance {between}, expected 1.5625"
t, v0, v1, between = best_split([1, 1, 0, 0, 0, 1, 1])
assert t == 2, f"cutoff {t}: on the waits 0, 1, 5, 6 every cutoff from 2 to 5 splits off 0 and 1, so return 2"
assert np.isclose(v0, 0.25) and np.isclose(v1, 0.25), f"group variances {v0}, {v1}: the groups are 0, 1 and 5, 6"
t, v0, v1, between = best_split(counts)
_, _, total = pmf_mean_var(counts)
w0 = counts[:t].sum() / counts.sum()
within = w0 * v0 + (1 - w0) * v1
assert t == 13, f"cutoff {t} on the calls, expected 13"
assert np.isclose(within + between, total), f"within + between = {within + between:.4f}, but the variance of all the waits is {total:.4f}"
print(f"best cutoff: waits below {t} minutes form the short group ({w0:.0%} of the calls)")
print(f"variance inside the short group {v0:.2f}, inside the long group {v1:.2f}")
print(f"within-group variance {within:.2f}, between-group {between:.2f}, total {total:.2f}")
print(f"the cutoff explains {between / total:.0%} of the variance of the waits")
print("Part 3 passed")

## Solution

### Part 1: Mean and variance from a histogram

In [ ]:
def pmf_mean_var(counts):
    """PMF, mean and variance of a value k that occurs counts[k] times."""
    counts = np.asarray(counts, dtype=float)
    values = np.arange(len(counts))
    pmf = counts / counts.sum()
    mean = values @ pmf
    var = ((values - mean) ** 2) @ pmf
    return pmf, float(mean), float(var)

**Why it works.** A wait of `k` minutes has probability `counts[k] / n`, so the histogram divided by
its total is the PMF. The mean and the variance are then sums over the values weighted by
the PMF, $\mu = \sum_k k\,p(k)$ and $\sigma^2 = \sum_k (k - \mu)^2 p(k)$, which is what the
two dot products compute. This is the variance of the 5,000 waits themselves, dividing by
$n$, the same number `np.var(waits)` gives.

### Part 2: Between-group variance at every cutoff

In [ ]:
def between_variance(counts):
    """b[t] = between-group variance when waits below t form one group, for t = 0..len(counts)."""
    counts = np.asarray(counts, dtype=float)
    values = np.arange(len(counts))
    n, total = counts.sum(), values @ counts
    n0 = np.r_[0.0, np.cumsum(counts)]               # calls below each cutoff
    s0 = np.r_[0.0, np.cumsum(values * counts)]      # their total wait
    n1, s1 = n - n0, total - s0                      # the same for the long group
    mean = total / n
    b = np.zeros(len(counts) + 1)
    both = (n0 > 0) & (n1 > 0)                       # cutoffs that leave neither group empty
    mu0, mu1 = s0[both] / n0[both], s1[both] / n1[both]
    b[both] = n0[both] / n * (mu0 - mean) ** 2 + n1[both] / n * (mu1 - mean) ** 2
    return b

**Why it works.** For a cutoff $t$ the short group is every wait below $t$, so its number of calls and
its total wait are running sums of `counts` and `values * counts` up to $t - 1$. A zero put
in front of each running sum lines entry $t$ up with the cutoff $t$. The long group gets
whatever the short one leaves of the totals, so one pass gives every cutoff. Dividing only
where both groups have calls keeps the empty splits at 0 without a division by zero.

The weights matter. Leaving them out treats a group of 10 calls like a group of 4,990, and
the cutoff that isolates a few extreme waits would look as good as a real split.

### Part 3: The best cutoff, and the variance left inside the groups

In [ ]:
def best_split(counts):
    """(t, var_short, var_long, between) at the cutoff with the largest between-group variance."""
    counts = np.asarray(counts, dtype=float)
    b = between_variance(counts)
    t = int(np.argmax(b))                            # argmax returns the first of tied maxima
    _, _, var_short = pmf_mean_var(counts[:t])       # waits 0 to t - 1
    _, _, var_long = pmf_mean_var(counts[t:])        # waits t and up, shifted down by t
    return t, var_short, var_long, float(b[t])

**Why it works.** By the law of total variance, the variance of all the waits is the within-group part
plus the between-group part, at every cutoff. The total does not depend on $t$, so the cutoff
with the largest between-group variance is also the one that leaves the least variance
inside the groups. A regression tree chooses its splits by this rule: predicting each group
by its mean leaves a mean squared error equal to the within-group variance.

`pmf_mean_var(counts[t:])` reads the waits $t, t + 1, \dots$ as $0, 1, \dots$. That moves
every value of the long group down by $t$, which changes its mean but not its variance, so
Part 1's function serves both groups as it is.

On the calls the best cutoff is 13 minutes. It explains 89% of the variance of the waits,
76.7 of 86.1. The within-group variance left is 9.4, from standard deviations of about 2.3
minutes among the short waits and 4.4 minutes among the long ones.